# Seeing Machines — Screenshot Companion
**CompSci for Designers 2 — Final Project**

**Status:** a complete, clean **Level 1** (CLIP joint-embedding retrieval), with a genuine
**Level 2** (VLM captioning → caption-grounded retrieval → grounded answers) layered on top
as its own independent route — not a relabeled Level 1.

**Corpus convention:** `ROOT/<steam_appid>/screenshots/*.jpg|png`. The App ID folder name is
ground-truth metadata, resolved to a real game name **once, offline** (see Section 2).

**Reproducibility:** this notebook makes **no third-party network calls** in its default
configuration, and depends on **no one's personal Google Drive**. `USE_DRIVE = False`
(default) runs entirely off a small included cache + sample-image bundle, so it executes
top-to-bottom on a fresh runtime for anyone — including a grader with no access to your
Drive. `USE_DRIVE = True` is for your own development against your full personal archive,
and is the *only* mode that ever talks to Steam's API (to resolve names for new app ids,
once, then cached to a file you can fold back into the shipped bundle).

This notebook builds:
1. Corpus scanning + a pre-resolved App ID → game name lookup (no live API call by default).
2. **Level 1**: CLIP joint embedding space — semantic text search, exact metadata filtering,
   and image-upload game identification.
3. **Level 2**: Gemma 3 4B generates a structured description of every image; descriptions
   are embedded as text and retrieved independently of Level 1, with a grounded
   natural-language answer on top.
4. A Gradio interface that runs either route, or both side by side for comparison.

**Runtime tip:** CPU is fine through corpus scanning. Switch to a T4 GPU for the CLIP
embedding cell, and again for the Gemma captioning cell — then back to CPU. Idling with a
GPU attached burns your free-tier quota.

In [ ]:
import os, zipfile
from pathlib import Path

# ── Why this exists ────────────────────────────────────────────────────────
# A hardcoded personal path like /content/drive/MyDrive/... only ever resolves
# for the account that mounted it. Sharing the folder does NOT make it appear
# at that path for anyone else — so a notebook built around it can never
# actually run "top to bottom on a fresh runtime" for a grader, regardless of
# Drive sharing settings. Defaulting to a small, included, self-contained
# bundle sidesteps the problem entirely instead of trying to work around it.

USE_DRIVE = False
# True  -> mount YOUR Drive, work against your full personal archive (dev use).
# False -> self-contained mode: needs only the cache_bundle.zip and
#          sample_corpus.zip included with this submission. No Drive, no
#          network calls, no dependency on anyone's account but your own
#          upload dialog below.

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT_DIR  = "/content/drive/MyDrive/SeeingMachines/screenshots"
    CACHE_DIR = "/content/drive/MyDrive/SeeingMachines/cache"
    os.makedirs(CACHE_DIR, exist_ok=True)
else:
    ROOT_DIR  = "/content/sample_corpus"
    CACHE_DIR = "/content/cache"
    os.makedirs(ROOT_DIR, exist_ok=True)
    os.makedirs(CACHE_DIR, exist_ok=True)

    def _ensure_unzipped(target_dir: str, label: str):
        if any(Path(target_dir).iterdir()):
            return
        from google.colab import files
        print(f"{target_dir} is empty — upload {label}:")
        uploaded = files.upload()
        zip_path = next(iter(uploaded))
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(target_dir)
        print(f"Extracted into {target_dir}")

    _ensure_unzipped(CACHE_DIR, "cache_bundle.zip (appid_to_name.json, embeddings, captions, corpus_index.csv)")
    _ensure_unzipped(ROOT_DIR, "sample_corpus.zip (representative image sample, ROOT/<appid>/screenshots/* layout)")

print(f"Mode: {'Drive (personal, full archive)' if USE_DRIVE else 'Self-contained (cache + sample bundle)'}")
print("ROOT_DIR:", ROOT_DIR, "| exists:", os.path.isdir(ROOT_DIR))
print("CACHE_DIR:", CACHE_DIR)

In [ ]:
!pip install -q open_clip_torch pillow pandas tqdm gradio requests
!pip install -q -U transformers accelerate bitsandbytes

## 1. Scan the corpus
Walks `ROOT/<appid>/...` and records every image file found under each numeric app-id
folder, regardless of how deep the `screenshots` subfolder sits.

In [ ]:
import re
from pathlib import Path
import pandas as pd

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def scan_corpus(root_dir: str) -> pd.DataFrame:
    rows = []
    root = Path(root_dir)
    for appid_dir in sorted(root.iterdir()):
        if not appid_dir.is_dir() or not re.fullmatch(r"\d+", appid_dir.name):
            continue
        appid = appid_dir.name
        for f in appid_dir.rglob("*"):
            if f.is_file() and f.suffix.lower() in IMG_EXTS:
                rows.append({"appid": appid, "filepath": str(f), "filename": f.name})
    df = pd.DataFrame(rows).drop_duplicates(subset="filepath").reset_index(drop=True)
    return df

corpus_df = scan_corpus(ROOT_DIR)
print(f"Found {len(corpus_df)} images across {corpus_df['appid'].nunique()} app-id folders.")

# The brief targets 50-150 images for L1/L2 (300 is the L3 ceiling). This isn't
# a hard stop, but captioning time scales directly with image count, and a
# corpus several times over spec risks both Colab session limits and scope
# creep beyond what the rubric is actually asking for.
if len(corpus_df) > 300:
    print(f"\n\u26a0\ufe0f  {len(corpus_df)} images exceeds even the Level 3 cap of 300 "
          f"(target is 50\u2013150 for L1/L2). Worth curating down before the captioning "
          f"step below \u2014 that's explicitly part of the L1 deliverable, not a detour.")
elif len(corpus_df) > 150:
    print(f"\nNote: {len(corpus_df)} images is above the L1/L2 target of 50\u2013150 "
          f"(within Level 3's 300 cap, if that's the plan). Worth a deliberate, documented "
          f"curation choice either way.")

corpus_df.head()

## 2. App ID → game name
**Reproducibility fix:** the previous version of this notebook called Steam's live API
during a normal run. A graded notebook shouldn't depend on a third-party service being up,
unrate-limited, and unchanged — so by default this cell makes **zero network calls** and
just loads `appid_to_name.json`, a small static file included in the cache bundle.

The live-fetch logic still exists, but only runs when `USE_DRIVE = True` — i.e. only when
you're actively developing against your own archive and have added app ids that aren't in
the cache yet. It hits the public, unauthenticated `store.steampowered.com/api/appdetails`
endpoint (no key required), once per new id, and folds the result straight into the same
cache file so it ships with the next bundle.

In [ ]:
import json, time
from pathlib import Path

RESOLVED_CACHE = Path(CACHE_DIR) / "appid_to_name.json"

if RESOLVED_CACHE.exists():
    with open(RESOLVED_CACHE) as f:
        appid_to_name = json.load(f)
    print(f"Loaded {len(appid_to_name)} pre-resolved app id -> name pairs from {RESOLVED_CACHE}")
else:
    appid_to_name = {}
    print(f"\u26a0\ufe0f  {RESOLVED_CACHE} not found.")
    if USE_DRIVE:
        print("Run the dev-only resolution cell below to build it from scratch.")
    else:
        print("This file should have shipped inside cache_bundle.zip. Without it, "
              "game_name will be empty for everything below.")

In [ ]:
# Dev-only — never runs during grading, since USE_DRIVE defaults to False and
# appid_to_name.json above already ships as a complete, included artifact.
if USE_DRIVE:
    import requests
    from tqdm.auto import tqdm

    unique_appids = sorted(corpus_df["appid"].unique(), key=int)

    def fetch_name(appid: str, max_retries: int = 5):
        """Look up one app id via the public, unauthenticated storefront API."""
        for _ in range(max_retries):
            try:
                r = requests.get("https://store.steampowered.com/api/appdetails",
                                  params={"appids": appid}, timeout=10)
            except requests.RequestException:
                time.sleep(5); continue
            if r.status_code == 429:
                time.sleep(10); continue
            if r.status_code == 403:
                time.sleep(300); continue
            if r.status_code != 200:
                return None
            data = r.json().get(str(appid), {})
            return data["data"]["name"] if data.get("success") else None
        return None

    unresolved_appids = [a for a in unique_appids if a not in appid_to_name]
    if unresolved_appids:
        print(f"Resolving {len(unresolved_appids)} new app id(s) via the live Steam API...")
        for appid in tqdm(unresolved_appids, desc="Resolving appids"):
            appid_to_name[appid] = fetch_name(appid) or f"Unknown game (appid {appid})"
            with open(RESOLVED_CACHE, "w") as f:
                json.dump(appid_to_name, f, indent=2, ensure_ascii=False)
            time.sleep(1.5)  # stay under ~200 req / 5 min
        print("Done. Fold the updated appid_to_name.json back into cache_bundle.zip "
              "before you next share/submit.")
    else:
        print("Nothing new to resolve \u2014 every scanned app id is already cached.")
else:
    print("Skipped (USE_DRIVE is False) \u2014 using the included appid_to_name.json as-is.")

In [ ]:
corpus_df["game_name"] = corpus_df["appid"].map(appid_to_name)

missing = corpus_df[corpus_df["game_name"].isna()]
if len(missing):
    print(f"\u26a0\ufe0f  {missing['appid'].nunique()} appid(s) have no entry in appid_to_name.json: "
          f"{sorted(missing['appid'].unique(), key=int)}")
    if USE_DRIVE:
        print("Run the resolution cell above again \u2014 it picks these up automatically.")
    else:
        print("Unexpected in self-contained mode \u2014 your cache bundle may be stale "
              "relative to sample_corpus.zip.")

corpus_index_path = Path(CACHE_DIR) / "corpus_index.csv"
corpus_df.to_csv(corpus_index_path, index=False)

print("\nImages per game (this is your corpus statement's raw material):")
print(corpus_df.groupby("game_name").size().sort_values(ascending=False))

## 3. Level 1 — joint embedding space (CLIP)
Switch to a GPU runtime now. Per your professor's read: this similarity-matching mechanism
on its own is already a solid, complete Level 1 — the goal here is a clean, well-documented
foundation, not more tuning.

In [ ]:
import torch, open_clip
from PIL import Image
from tqdm.auto import tqdm
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model = clip_model.to(device).eval()

In [ ]:
EMB_PATH = Path(CACHE_DIR) / "clip_embeddings.npy"
EMB_INDEX_PATH = Path(CACHE_DIR) / "clip_embeddings_index.csv"

def embed_corpus(df: pd.DataFrame, force_recompute: bool = False):
    if EMB_PATH.exists() and EMB_INDEX_PATH.exists() and not force_recompute:
        print("Loading cached embeddings...")
        return np.load(EMB_PATH), pd.read_csv(EMB_INDEX_PATH)

    feats = []
    with torch.no_grad():
        for fp in tqdm(df["filepath"], desc="Embedding images"):
            img = clip_preprocess(Image.open(fp).convert("RGB")).unsqueeze(0).to(device)
            feat = clip_model.encode_image(img)
            feat = feat / feat.norm(dim=-1, keepdim=True)
            feats.append(feat.cpu().numpy()[0])
    feats = np.stack(feats)
    np.save(EMB_PATH, feats)
    df.to_csv(EMB_INDEX_PATH, index=False)
    print(f"Cached {len(feats)} embeddings to {EMB_PATH}")
    return feats, df

clip_embeddings, indexed_df = embed_corpus(corpus_df)

## 4. Level 1 retrieval logic
- **`embed_text` + `cosine_topk`** — descriptive/semantic queries.
- **`find_named_game` + exact filter** — "pull up all screenshots from \<game\>": a literal
  metadata lookup, since your folder structure already gives you a ground-truth answer.
- **`embed_query_image`** — "what game is this?" for an unsorted screenshot: nearest-neighbor
  classification against your indexed corpus, reported with a similarity score.

This is pure CLIP image-embedding similarity — no captions involved yet. Its blind spots
(e.g. a query like "fish" failing to surface a screenshot that merely *contains* a fish in
an otherwise unrelated scene) are exactly what Level 2, below, is built to address.

In [ ]:
import difflib

def embed_text(query: str) -> np.ndarray:
    with torch.no_grad():
        tok = clip_tokenizer([query]).to(device)
        feat = clip_model.encode_text(tok)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy()[0]

def embed_query_image(pil_img: Image.Image) -> np.ndarray:
    with torch.no_grad():
        img = clip_preprocess(pil_img.convert("RGB")).unsqueeze(0).to(device)
        feat = clip_model.encode_image(img)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy()[0]

def cosine_topk(query_vec: np.ndarray, top_k: int = 8, min_sim: float = 0.0) -> pd.DataFrame:
    sims = clip_embeddings @ query_vec
    order = np.argsort(-sims)[:top_k]
    results = indexed_df.iloc[order].copy()
    results["similarity"] = sims[order]
    return results[results["similarity"] >= min_sim]

def find_named_game(query: str):
    """Fuzzy-match a query against known game names in the corpus."""
    names = indexed_df["game_name"].dropna().unique().tolist()
    lowered = [n.lower() for n in names]
    match = difflib.get_close_matches(query.lower(), lowered, n=1, cutoff=0.6)
    if match:
        return names[lowered.index(match[0])]
    for n in names:
        if n.lower() in query.lower() or query.lower() in n.lower():
            return n
    return None

def search(query_text: str = None, query_image: Image.Image = None,
           top_k: int = 8, min_sim: float = 0.0):
    """Level 1 search: pure CLIP image-embedding similarity."""
    if query_image is not None:
        vec = embed_query_image(query_image)
        results = cosine_topk(vec, top_k=top_k, min_sim=min_sim)
        if len(results):
            best = results.iloc[0]
            answer = f"Best guess: **{best['game_name']}** (similarity {best['similarity']:.2f}, appid {best['appid']})"
        else:
            answer = "No confident match found."
        return answer, results

    if query_text:
        named = find_named_game(query_text)
        wants_all = any(w in query_text.lower() for w in ["all", "every", "pull up", "show me"])
        if named and wants_all:
            results = indexed_df[indexed_df["game_name"] == named].copy()
            results["similarity"] = 1.0
            answer = f"Showing all {len(results)} screenshots tagged **{named}** (exact metadata match)."
            return answer, results
        vec = embed_text(query_text)
        results = cosine_topk(vec, top_k=top_k, min_sim=min_sim)
        answer = f"Top {len(results)} semantic matches for: \"{query_text}\" (Level 1, CLIP)"
        return answer, results

    return "Enter a text query or upload an image.", indexed_df.iloc[0:0]

## 5. Level 2 — Gemma 3 4B captioning

**Schema** (one JSON object per image — this is the design decision the brief calls out as
central to the level; reshape it if you want, but here's the reasoning behind this version):

```
{
  "scene_type": "menu_ui | combat | exploration | cutscene_dialogue | inventory_crafting | map_screen | other",
  "visible_elements": ["short noun phrases for distinct objects/creatures/items actually visible"],
  "setting_description": "1-2 sentences: environment and any action",
  "visible_text": "on-screen HUD/UI text, verbatim if legible, else empty string",
  "guessed_genre_or_title": "the model's OWN blind guess \u2014 it is never told the real game name",
  "confidence_notes": "anything the model itself flags as uncertain"
}
```

Why `visible_elements` is its own list rather than folded into the free-text description:
it's the field that makes Level 2 actually "see" things Level 1's holistic image embedding
tends to skate past. A screenshot's CLIP embedding represents the whole frame at once, so a
small but real detail \u2014 a fish in a tank in the corner of an otherwise ordinary room \u2014
can get diluted into "indoor scene" and never surface for a query like "fish." An explicit
object list, embedded and searched as text, lets that one word actually be there to match
against, even when it wasn't visually dominant.

Why `guessed_genre_or_title` is kept separate from ground truth: the model is **never told**
the real game name. Comparing this blind guess against the `game_name` you already resolved
from the App ID *is* your mis-seeing dossier \u2014 every mismatch is a documented,
mechanistically explainable failure, for free.

**Why `bnb_4bit_compute_dtype=torch.float32` below, not bfloat16 or float16:** the brief
flags both as unstable for Gemma 3 on a T4 specifically. float32 compute is slower, but this
runs once and gets cached \u2014 stability matters more than speed here, and the tradeoff
itself is the kind of parameter decision worth a line in your write-up.

In [ ]:
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig

VLM_ID = "google/gemma-3-4b-it"

vlm_quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32,  # see markdown above \u2014 deliberately not bf16/fp16 on T4
)

vlm_model = Gemma3ForConditionalGeneration.from_pretrained(
    VLM_ID,
    device_map="auto",
    quantization_config=vlm_quant_config,
).eval()
vlm_processor = AutoProcessor.from_pretrained(VLM_ID)

print(f"Loaded {VLM_ID} in 4-bit. Memory footprint: {vlm_model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
CAPTION_SYSTEM_PROMPT = (
    "You analyze a single video game screenshot. Describe ONLY what is visibly present \u2014 "
    "do not use any outside knowledge of specific game titles. Respond with STRICT JSON only, "
    "no markdown fences, no commentary outside the JSON, matching exactly this schema:\n"
    '{"scene_type": "menu_ui|combat|exploration|cutscene_dialogue|inventory_crafting|map_screen|other", '
    '"visible_elements": ["short noun phrases for distinct objects, creatures, or items you can see"], '
    '"setting_description": "1-2 sentences describing the environment and any action", '
    '"visible_text": "any on-screen HUD or UI text you can read, verbatim, or empty string", '
    '"guessed_genre_or_title": "your own blind guess at genre or game title, based only on visuals", '
    '"confidence_notes": "anything you are unsure about"}'
)

def caption_image(pil_img: Image.Image, max_new_tokens: int = 300) -> dict:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": CAPTION_SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": pil_img},
            {"type": "text", "text": "Describe this screenshot."},
        ]},
    ]
    inputs = vlm_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(vlm_model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        out = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    raw = vlm_processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

    cleaned = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw_text": raw}

In [ ]:
CAPTION_CACHE = Path(CACHE_DIR) / "captions.json"
CAPTION_LIMIT = None  # set to a small int (e.g. 10) to test before committing to a full run

def load_captions() -> dict:
    if CAPTION_CACHE.exists():
        with open(CAPTION_CACHE) as f:
            return json.load(f)
    return {}

def save_captions(d: dict):
    with open(CAPTION_CACHE, "w") as f:
        json.dump(d, f, indent=2, ensure_ascii=False)

captions = load_captions()
to_caption = [r for r in indexed_df.itertuples() if r.filepath not in captions]
if CAPTION_LIMIT is not None:
    to_caption = to_caption[:CAPTION_LIMIT]

print(f"{len(captions)} cached, {len(to_caption)} screenshot(s) left to caption this run.")

for row in tqdm(to_caption, desc="Captioning"):
    img = Image.open(row.filepath).convert("RGB")
    captions[row.filepath] = caption_image(img)
    save_captions(captions)  # one image at a time \u2014 a disconnect costs nothing

parse_errors = [fp for fp, c in captions.items() if c.get("parse_error")]
if parse_errors:
    print(f"\n\u26a0\ufe0f  {len(parse_errors)} caption(s) weren\'t valid JSON \u2014 worth a look "
          f"for your mis-seeing dossier (the model failing to follow format is itself a finding).")

## 6. Level 2 retrieval + grounded answers
Captions are flattened to a single string per image and embedded with the **same CLIP text
encoder** already loaded for Level 1 \u2014 a deliberate simplicity choice (one model stack,
not two) rather than adding a dedicated sentence-embedding model. If you find this caps how
far the associative jumps reach (e.g. "fish" not landing near "lake"), swapping in something
like a small sentence-transformer here is a contained, one-function change worth trying and
documenting as a comparison.

The grounded answer step feeds the retrieved captions \u2014 not the raw images \u2014 back into
Gemma as text context and asks it to answer using only what's there. (Passing the actual
retrieved images into the VLM's context instead is Level 3's move, not this one.)

In [ ]:
def caption_to_text(c: dict) -> str:
    if c.get("parse_error"):
        return c.get("raw_text", "")
    elements = ", ".join(c.get("visible_elements", []))
    return (
        f"{c.get('setting_description', '')} "
        f"Visible elements: {elements}. "
        f"Scene type: {c.get('scene_type', '')}. "
        f"On-screen text: {c.get('visible_text', '')}"
    ).strip()

indexed_df["caption_text"] = indexed_df["filepath"].map(lambda fp: caption_to_text(captions.get(fp, {})))
indexed_df["guessed_genre_or_title"] = indexed_df["filepath"].map(lambda fp: captions.get(fp, {}).get("guessed_genre_or_title", ""))

CAPTION_EMB_PATH = Path(CACHE_DIR) / "caption_embeddings.npy"

def embed_captions(df: pd.DataFrame, force_recompute: bool = False) -> np.ndarray:
    if CAPTION_EMB_PATH.exists() and not force_recompute:
        return np.load(CAPTION_EMB_PATH)
    embs = []
    with torch.no_grad():
        for text in tqdm(df["caption_text"], desc="Embedding captions"):
            tok = clip_tokenizer([text or " "]).to(device)
            feat = clip_model.encode_text(tok)
            feat = feat / feat.norm(dim=-1, keepdim=True)
            embs.append(feat.cpu().numpy()[0])
    embs = np.stack(embs)
    np.save(CAPTION_EMB_PATH, embs)
    return embs

caption_embeddings = embed_captions(indexed_df)

def search_captions(query_text: str, top_k: int = 8, min_sim: float = 0.0) -> pd.DataFrame:
    """Level 2 search: caption-text embedding similarity."""
    vec = embed_text(query_text)
    sims = caption_embeddings @ vec
    order = np.argsort(-sims)[:top_k]
    results = indexed_df.iloc[order].copy()
    results["similarity"] = sims[order]
    return results[results["similarity"] >= min_sim]

In [ ]:
def generate_grounded_answer(query: str, retrieved_rows: pd.DataFrame, max_new_tokens: int = 200) -> str:
    if len(retrieved_rows) == 0:
        return "No matching screenshots found in the archive for this query."

    context_lines = [
        f"- [{row.game_name}] {row.caption_text}" for row in retrieved_rows.itertuples()
    ]
    context = "\n".join(context_lines)

    messages = [
        {"role": "system", "content": [{"type": "text", "text":
            "Answer the question using ONLY the screenshot descriptions provided below. "
            "If they don't support an answer, say so plainly instead of guessing."}]},
        {"role": "user", "content": [{"type": "text", "text":
            f"Screenshot descriptions:\n{context}\n\nQuestion: {query}"}]},
    ]
    inputs = vlm_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(vlm_model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        out = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return vlm_processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

## 7. Level 1 vs Level 2 — side-by-side comparison
The brief requires this on at least 5 shared queries. `"fish"` is here on purpose \u2014 it's
the exact case Level 2 exists to handle: a query with no dedicated screenshots, where the
useful answer is "the closest related thing," not "nothing." Add your own queries and write
up *why* the two routes diverge where they do \u2014 that interpretation is the actual
deliverable, this just generates the raw comparison.

In [ ]:
comparison_queries = [
    "fish",
    "a nighttime city street",
    "a health bar or HUD element",
    "an inventory or crafting menu",
    "a boss fight",
]

for q in comparison_queries:
    print(f"\n=== Query: {q!r} ===")
    l1 = cosine_topk(embed_text(q), top_k=5)
    l2 = search_captions(q, top_k=5)
    print("Level 1 (CLIP image embeddings):")
    print(l1[["game_name", "filename", "similarity"]].to_string(index=False))
    print("Level 2 (caption embeddings):")
    print(l2[["game_name", "filename", "similarity"]].to_string(index=False))

## 8. Gradio interface
Functional, not polished — per the brief, interface hours go to the pipeline, not the
frontend. The route selector is what makes the L1/L2 comparison tangible instead of buried
in printed output.

In [ ]:
import gradio as gr

def gradio_search(query_text, query_image_path, route):
    if query_image_path:
        pil_img = Image.open(query_image_path).convert("RGB")
        answer, results = search(query_image=pil_img, top_k=12)
        gallery = [(r["filepath"], f"{r['game_name']} \u00b7 {r.get('similarity', 0):.2f}")
                   for _, r in results.iterrows()]
        return answer, gallery

    if not query_text:
        return "Enter a text query or upload an image.", []

    if route == "Level 1 \u2014 CLIP image similarity":
        answer, results = search(query_text=query_text, top_k=12)

    elif route == "Level 2 \u2014 caption-grounded":
        results = search_captions(query_text, top_k=8)
        answer = generate_grounded_answer(query_text, results)

    else:  # both, side by side
        l1_results = search(query_text=query_text, top_k=6)[1]
        l2_results = search_captions(query_text, top_k=6)
        grounded = generate_grounded_answer(query_text, l2_results)
        results = pd.concat([
            l1_results.assign(route="L1"),
            l2_results.assign(route="L2"),
        ])
        answer = f"**Level 2 grounded answer:** {grounded}"

    gallery = [(r["filepath"], f"{r['game_name']} \u00b7 {r.get('similarity', 0):.2f}")
               for _, r in results.iterrows()]
    return answer, gallery

with gr.Blocks(title="Seeing Machines \u2014 Screenshot Companion") as demo:
    gr.Markdown("# Seeing Machines: Screenshot Companion")
    with gr.Row():
        txt = gr.Textbox(label="Ask something", placeholder='e.g. "fish" or "a nighttime city street"')
        img = gr.Image(label="...or upload a screenshot to identify", type="filepath")
    route = gr.Radio(
        ["Level 1 \u2014 CLIP image similarity", "Level 2 \u2014 caption-grounded", "Both side-by-side"],
        value="Level 2 \u2014 caption-grounded", label="Retrieval route",
    )
    btn = gr.Button("Search")
    answer_box = gr.Markdown()
    gallery = gr.Gallery(label="Results", columns=4, height=500)
    btn.click(gradio_search, inputs=[txt, img, route], outputs=[answer_box, gallery])

demo.launch(share=True)